# 08 - Experimentation with MLflow and Error Analysis (Objective 7)
- Part A: classic models - compare **preprocessing x features x model** (fast).
- Part B: BERT experiments - compare learning rate / sequence length / model (slow, needs GPU).
- Part C: compare all runs.
- Part D: error analysis of misclassified reviews.

Rules: change **one thing at a time**, use the **same splits and seeds**, evaluate on **validation**.

In [ ]:
%pip install -q mlflow scikit-learn pandas transformers datasets accelerate torch pyarrow

In [ ]:
import mlflow
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

mlflow.set_tracking_uri("file:./mlruns")           # or "http://localhost:5000" if you run `mlflow ui`/server
mlflow.set_experiment("montra_sentiment")

train = pd.read_parquet("data/processed/train.parquet")
val = pd.read_parquet("data/processed/val.parquet")
test = pd.read_parquet("data/processed/test.parquet")
train["text_raw"], val["text_raw"] = train["review_text"], val["review_text"]

## Part A - Classic model experiments

In [ ]:
TEXT_COLUMNS = {"raw": "text_raw", "light": "text_light", "heavy": "text_heavy"}   # preprocessing strategies
FEATURES = {"tfidf_1gram": lambda: TfidfVectorizer(ngram_range=(1, 1), min_df=2),
            "tfidf_1-2gram": lambda: TfidfVectorizer(ngram_range=(1, 2), min_df=2),
            "bow_1-2gram": lambda: CountVectorizer(ngram_range=(1, 2), min_df=2)}
MODELS = {"logreg": lambda: LogisticRegression(max_iter=1000, class_weight="balanced"),
          "svm": lambda: LinearSVC(class_weight="balanced"),
          "naive_bayes": lambda: MultinomialNB()}

for prep, col in TEXT_COLUMNS.items():
    for feat_name, make_feat in FEATURES.items():
        for model_name, make_model in MODELS.items():
            with mlflow.start_run(run_name=f"{model_name}|{feat_name}|{prep}"):
                mlflow.set_tags({"family": "classic", "preprocessing": prep})
                mlflow.log_params({"model": model_name, "features": feat_name, "preprocessing": prep, "seed": 42})

                pipe = make_pipeline(make_feat(), make_model())
                pipe.fit(train[col], train["label"])
                pred = pipe.predict(val[col])

                mlflow.log_metrics({"val_accuracy": accuracy_score(val["label"], pred),
                                    "val_f1_macro": f1_score(val["label"], pred, average="macro")})

                fig, ax = plt.subplots(figsize=(4, 4))
                ConfusionMatrixDisplay.from_predictions(val["label"], pred, ax=ax, colorbar=False)
                mlflow.log_figure(fig, "confusion_matrix.png"); plt.close(fig)
print("Part A finished")

## Part B - BERT experiments
Each run trains a model and logs it to MLflow. Start with 2 configurations and add more later.

In [ ]:
import torch
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments,
                          Trainer, DataCollatorWithPadding, set_seed)

label2id = {"negative": 0, "neutral": 1, "positive": 2}
id2label = {v: k for k, v in label2id.items()}

class DS(torch.utils.data.Dataset):
    def __init__(self, df, tok, max_len):
        self.enc = tok(df["text_light"].tolist(), truncation=True, max_length=max_len)
        self.y = df["label"].map(label2id).tolist()
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        d = {k: v[i] for k, v in self.enc.items()}; d["labels"] = self.y[i]; return d

def metrics(p):
    preds = p.predictions.argmax(-1)
    return {"accuracy": accuracy_score(p.label_ids, preds), "f1_macro": f1_score(p.label_ids, preds, average="macro")}

def run_bert(cfg):
    set_seed(42)
    tok = AutoTokenizer.from_pretrained(cfg["model"])
    model = AutoModelForSequenceClassification.from_pretrained(cfg["model"], num_labels=3, id2label=id2label, label2id=label2id)
    args = TrainingArguments(output_dir="models/tmp", num_train_epochs=cfg["epochs"], learning_rate=cfg["lr"],
                             per_device_train_batch_size=cfg["batch"], per_device_eval_batch_size=32,
                             eval_strategy="epoch", save_strategy="no", seed=42, report_to="none")
    trainer = Trainer(model=model, args=args, train_dataset=DS(train, tok, cfg["max_len"]),
                      eval_dataset=DS(val, tok, cfg["max_len"]), data_collator=DataCollatorWithPadding(tok),
                      compute_metrics=metrics)
    with mlflow.start_run(run_name=f"{cfg['model']}|lr{cfg['lr']}|len{cfg['max_len']}|ep{cfg['epochs']}"):
        mlflow.set_tag("family", "bert")
        mlflow.log_params(cfg)
        trainer.train()
        res = trainer.evaluate()
        mlflow.log_metrics({"val_accuracy": res["eval_accuracy"], "val_f1_macro": res["eval_f1_macro"]})

BERT_CONFIGS = [
    {"model": "distilbert-base-uncased", "lr": 3e-5, "max_len": 128, "epochs": 3, "batch": 16},
    {"model": "distilbert-base-uncased", "lr": 5e-5, "max_len": 128, "epochs": 3, "batch": 16},
    # {"model": "distilbert-base-uncased", "lr": 3e-5, "max_len": 256, "epochs": 3, "batch": 16},
    # {"model": "bert-base-uncased",       "lr": 3e-5, "max_len": 128, "epochs": 3, "batch": 16},
    # {"model": "roberta-base",            "lr": 2e-5, "max_len": 128, "epochs": 3, "batch": 16},
]
for cfg in BERT_CONFIGS:
    run_bert(cfg)

## Part C - Compare all runs

In [ ]:
runs = mlflow.search_runs(experiment_names=["montra_sentiment"], order_by=["metrics.val_f1_macro DESC"])
cols = ["tags.mlflow.runName", "tags.family", "metrics.val_accuracy", "metrics.val_f1_macro"]
runs[cols].head(15).round(3)
# Browse visually:  run `mlflow ui` in a terminal, then open http://localhost:5000

In [ ]:
# Which preprocessing strategy works best on average? (classic models)
classic = runs[runs["tags.family"] == "classic"]
classic.groupby("params.preprocessing")["metrics.val_f1_macro"].mean().sort_values(ascending=False).round(3)

## Part D - Error analysis
Look at the BERT predictions on the test set (from notebook 06) and tag *why* it failed.
Rule-based tags are a quick first pass; confirm by reading examples.

In [ ]:
preds = pd.read_parquet("data/outputs/bert_test_predictions.parquet")
errors = preds[preds["label"] != preds["pred"]].copy()
print(f"Errors: {len(errors)} of {len(preds)} ({len(errors)/len(preds):.1%})")

NEGATIONS = ["not ", "no ", "never ", "n't", "nothing", "without"]
CONTRAST = [" but ", " however ", " although ", " though ", " except "]
ASPECTS = ["staff", "room", "food", "breakfast", "location", "price", "pool", "gym", "clean", "bed", "wifi"]

def tag_error(row):
    t = " " + row["text_light"].lower() + " "
    tags = []
    if any(c in t for c in CONTRAST): tags.append("mixed_sentiment")
    if any(n in t for n in NEGATIONS): tags.append("negation")
    if sum(a in t for a in ASPECTS) >= 3: tags.append("multiple_topics")
    if row["label"] == "neutral" or row["pred"] == "neutral": tags.append("neutral_ambiguity")
    if len(t.split()) < 8: tags.append("very_short")
    return tags or ["other"]

errors["tags"] = errors.apply(tag_error, axis=1)
tag_counts = errors["tags"].explode().value_counts()
tag_counts

In [ ]:
tag_counts.plot.bar(title="Error categories (a review can have several tags)"); plt.ylabel("count"); plt.show()

# most common confusions
print(pd.crosstab(errors["label"], errors["pred"]))

# read real examples per category
for tag in tag_counts.index[:3]:
    print(f"\n=== {tag} ===")
    for _, r in errors[errors["tags"].apply(lambda x: tag in x)].head(3).iterrows():
        print(f"[actual={r['label']} / pred={r['pred']}] {r['text_light'][:200]}")

### What to write in your model evaluation report
1. Baseline vs. BERT table (accuracy, macro-F1) from notebook 06.
2. The MLflow experiment table (Part C), showing which preprocessing, features and hyperparameters mattered.
3. Error categories with counts and example reviews (Part D).
4. Limitations: rating-derived labels, small neutral class, sarcasm, multi-topic reviews.